# [7.4] Mini Natural Language Autoencoders - Solutions

This notebook follows the same path as the exercise notebook: build the phrase-bottleneck contracts, inspect the CPU report, inspect the committed CUDA result, then optionally run the live CUDA preflight.

<details>
<summary>Expected output</summary>

All section tests should pass, and the signature-result plot should show the NLA phrase bottleneck below text-only and prompt-label baselines.

</details>


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

chapter = "chapter7_activation_to_language"
section = "part4_mini_natural_language_autoencoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mini_natural_language_autoencoders.tests as tests
from chapter7_activation_to_language.exercises.part4_mini_natural_language_autoencoders import solutions

GT_TIER = "GT-3"
EXERCISE_ID = "7_4_mini_natural_language_autoencoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for the CUDA preflight"
REQUIRES_GPU = True


## Unit Contracts

Each test is intentionally close to the exercise it validates. This mirrors original ARENA’s rhythm: implement a function, run a local test, then interpret the result.

<details>
<summary>Help - why so many small tests?</summary>

The full CUDA report is too coarse for debugging. Small tests catch alignment, shape, threshold, and leakage mistakes before the student reaches the live model path.

</details>


In [ ]:
tests.test_build_nla_training_batch_validates_alignment(
    solutions.build_nla_training_batch,
)
tests.test_build_nla_training_batch_rejects_empty_batches(
    solutions.build_nla_training_batch,
)
tests.test_generated_explanations_do_not_hide_numeric_coefficients(
    solutions._numeric_literal_count,
)
tests.test_activation_reconstruction_report_beats_text_only_baseline(
    solutions.activation_reconstruction_report,
)
tests.test_activation_reconstruction_report_rejects_empty_and_rank1_inputs(
    solutions.activation_reconstruction_report,
)
tests.test_logit_diff_preservation_report_checks_actual_logit_diff(
    solutions.logit_diff_preservation_report,
)
tests.test_logit_diff_preservation_report_rejects_bad_inputs(
    solutions.logit_diff_preservation_report,
)
tests.test_latent_preservation_report_requires_accuracy_and_agreement(
    solutions.latent_preservation_report,
)
tests.test_latent_preservation_report_rejects_invalid_thresholds(
    solutions.latent_preservation_report,
)
tests.test_brevity_and_counterfactual_reports_reject_prompt_copying(
    solutions.generated_text_brevity_report,
    solutions.counterfactual_explanation_report,
)
tests.test_brevity_and_counterfactual_reports_reject_bad_controls(
    solutions.generated_text_brevity_report,
    solutions.counterfactual_explanation_report,
)
tests.test_trainable_discrete_bottleneck_learns_phrase_ids(
    solutions.train_discrete_nla_bottleneck,
)
tests.test_trainable_discrete_bottleneck_rejects_empty_splits(
    solutions.train_discrete_nla_bottleneck,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## CPU Contract

The CPU report should already show the logic of the section: reconstruction beats a text-only baseline, target logit diff is preserved, latent state is preserved, phrase text is shorter, the counterfactual phrase changes, and the tiny phrase bottleneck trains.

<details>
<summary>Expected output</summary>

`activation_mse` should be about `0.01`, logit-diff error about `0.15`, and the toy trainable bottleneck should reach `eval_phrase_accuracy = 1.0`.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["reconstruction"]["beats_text_only"]
assert 0.149 < contract["logit_diff"]["mean_abs_error"] < 0.151
assert contract["latent_preservation"]["preserves_latents"]
assert contract["brevity"]["shorter_than_original"]
assert contract["counterfactual"]["explanation_changed"]
assert contract["trainable_bottleneck"]["eval_phrase_accuracy"] == 1.0
assert contract["trainable_bottleneck"]["beats_blank_text"]
contract


## Signature Result

Now inspect the accepted CUDA report. The important visual result is not just `preflight_passed`; it is the ordering of reconstruction errors and controls.

<details>
<summary>Interpreting the signature result</summary>

The NLA bottleneck has lower MSE than text-only and prompt-label baselines, while shuffled phrase text is worse. That supports the phrase carrying activation-specific information. The latent/probe checks and counterfactual phrase change make the result harder to explain as pure reconstruction luck.

</details>


<details>
<summary>Common bug</summary>

Do not compare the NLA reconstruction only against a blank-text baseline. The prompt-label baseline is harder and catches cases where phrase text is just a thin label leak.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["activation_shape"] == [8, 512]
assert gpu["text_bottleneck"] == "discrete_natural_language_phrase_bottleneck"
assert gpu["live_trainable_nla"]
assert not gpu["report_replay"]
assert gpu["trainable_encoder_train_accuracy"] == 1.0
assert gpu["trainable_eval_phrase_accuracy"] >= 0.75
assert gpu["trainable_encoder_final_loss"] < 0.01
assert gpu["trainable_beats_blank_text"]
assert gpu["phrase_count"] == 12
assert gpu["numeric_literal_count"] == 0
assert gpu["activation_mse"] < gpu["text_only_mse"]
assert gpu["activation_mse"] < gpu["prompt_label_baseline_mse"]
assert gpu["mean_cosine_similarity"] >= 0.93
assert gpu["preserves_target_logit_diff"]
assert gpu["probe_logit_mean_abs_error"] <= 2.0
assert gpu["preserves_latents"]
assert gpu["text_only_prediction_accuracy"] == 0.5
assert gpu["nla_prediction_accuracy"] == 1.0
assert gpu["passes_ood"]
assert gpu["counterfactual_explanation_changed"]
assert gpu["shuffled_control_worse"]
assert gpu["blank_text_control_worse"]
assert gpu["within_vram_budget"]

fig, ax = plt.subplots(figsize=(6.5, 3.2))
names = ["NLA", "text-only", "prompt label", "shuffled"]
values = [
    gpu["activation_mse"],
    gpu["text_only_mse"],
    gpu["prompt_label_baseline_mse"],
    gpu["shuffled_reconstruction_mse"],
]
ax.bar(names, values, color=["#0ea5e9", "#94a3b8", "#f59e0b", "#ef4444"])
ax.set_ylabel("MSE to held-out residual")
ax.set_title("Mini NLA reconstruction beats text baselines")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

{key: gpu[key] for key in [
    "model_name",
    "text_bottleneck",
    "example_generated_explanation",
    "numeric_literal_count",
    "activation_mse",
    "text_only_mse",
    "prompt_label_baseline_mse",
    "mean_cosine_similarity",
    "probe_logit_mean_abs_error",
    "nla_prediction_accuracy",
    "text_only_prediction_accuracy",
    "counterfactual_explanation_changed",
    "blank_text_mse",
    "peak_vram_gb",
]}


## Live CUDA Path

The report above is a committed artifact. This cell runs the live CUDA preflight through `solutions.py`.

<details>
<summary>Expected output</summary>

`preflight_passed` should be `True`, peak VRAM should stay well below the 24GB budget, and the live metrics should match the same qualitative pattern as the committed report.

</details>


In [ ]:
def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["peak_vram_gb"] <= 24.0
{
    "preflight_passed": live_gpu["preflight_passed"],
    "activation_mse": round(live_gpu["activation_mse"], 4),
    "text_only_mse": round(live_gpu["text_only_mse"], 4),
    "nla_prediction_accuracy": live_gpu["nla_prediction_accuracy"],
    "text_only_prediction_accuracy": live_gpu["text_only_prediction_accuracy"],
    "peak_vram_gb": round(live_gpu["peak_vram_gb"], 3),
}


## Limitations

This is a GT-3 local mini-NLA preflight on one pinned `gelu-1l` hook and tiny safe prompt splits. It is not Anthropic-scale NLA training, not a free-form language decoder, not arbitrary activation explanation, and not hidden-thought recovery.

## Further Research

Try paraphrase-robust phrase banks, multiple residual layers, nonsense short-label controls, more seeds, downstream activation injection, and eventually a learned local text decoder with its own baselines.
